# KMeans — Regionais de Saúde de MG por Ano

Unidade de agrupamento: par **(REGIONAL DE SAÚDE, ANO)**. Uma mesma Regional pode cair em agrupamentos diferentes em anos diferentes

O notebook está dividido em duas partes: 
**Parte 1 — Implementação** (entrada, parâmetros, execução do algoritmo e testes de robustez da configuração) 
**Parte 2 — Avaliação e Apresentação** (só resultados: gráficos e caracterização de cada cluster, sem repetir o método).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

sys.path.append(str(Path.cwd()))

from utilitarios_agrupamento_kmeans import (
    COLUNA_ANO, COLUNA_REGIONAL,
    BLOCOS_DE_TAXA, PASTA_TAXA,
    LIMIAR_MAXIMO_DE_NULOS_POR_COLUNA,
    USAR_PCA, RAZAO_MAXIMA_ATRIBUTOS_POR_LINHA, VARIANCIA_EXPLICADA_ALVO,
    NUMERO_MAXIMO_DE_COMPONENTES, SEMENTE_ALEATORIA,
    USAR_JANELA_COMUM, ANO_INICIAL_COMUM, ANO_FINAL_COMUM,
    K_MINIMO, K_MAXIMO, SEMENTES_DA_BUSCA, MINIMO_DE_ELEMENTOS_POR_CLUSTER, MAXIMO_DE_ITERACOES,
    NUMERO_DE_VARIAVEIS_DISCRIMINANTES,
    montar_painel,
    descartar_colunas_sem_informacao_e_total, descartar_colunas_com_muitos_nulos, tratar_nulos_sem_imputar,
    padronizar_e_reduzir,
    calcular_inercia_e_silhueta_por_k, escolher_k_pelo_cotovelo, busca_em_grade_k,
    maximo_de_iteracoes_usadas,
    executar_kmeans, resumir_agrupamento_kmeans, diagnostico_tamanho_dos_clusters,
    montar_resumo_parametros_kmeans, montar_tabela_pertencimento, montar_visao_agregada_regional_ano,
    montar_resumo_por_cluster,
    calcular_caracterizacao_por_cluster, selecionar_variaveis_mais_discriminantes,
    descrever_caracterizacao_por_cluster,
    calcular_parcelas_distancia_ao_centroide,
    registrar_log,
)
from graficos_agrupamento_kmeans import (
    grafico_cotovelo_inercia,
    grafico_distribuicao_por_regional, grafico_distribuicao_por_ano,
    grafico_heatmap_regional_ano,
    grafico_variaveis_discriminantes_por_cluster,
)

# Parte 1 — Implementação

Entrada dos dados, parâmetros de configuração, execução do algoritmo e os testes que validam a configuração escolhida. Nenhum resultado final é discutido aqui -- isso fica todo na Parte 2.

## 1. Leitura dos CSVs de taxa e reformatação 

In [2]:
painel = montar_painel(blocos=BLOCOS_DE_TAXA, pasta_taxa=PASTA_TAXA)
painel.head()

[LOG] tabela_casos_por_faixa_etária_taxa.csv: 9 categoria(s), 1064 par(es) regional-ano.
[LOG] tabela_casos_por_escolaridade_taxa.csv: 8 categoria(s), 728 par(es) regional-ano.
[LOG] tabela_casos_por_raça_e_cor_taxa.csv: 7 categoria(s), 728 par(es) regional-ano.
[LOG] tabela_mortes_por_faixa_etária_taxa.csv: 9 categoria(s), 1288 par(es) regional-ano.
[LOG] tabela_mamografias_por_faixa_etária_taxa.csv: 9 categoria(s), 364 par(es) regional-ano.
[LOG] Painel montado: 1316 par(es) regional-ano x 42 variável(is), 28 regional(is).


casos-faixa_0-9  casos-faixa_10-19  casos-faixa_20-29  \
REGIONAL DE SAÚDE ANO                                                           
alfenas           1979              NaN                NaN                NaN   
                  1980              NaN                NaN                NaN   
                  1981              NaN                NaN                NaN   
                  1982              NaN                NaN                NaN   
                  1983              NaN                NaN                NaN   

                        casos-faixa_30-39  casos-faixa_40-49  \
REGIONAL DE SAÚDE ANO                                          
alfenas           1979                NaN                NaN   
                  1980                NaN                NaN   
                  1981                NaN                NaN   
                  1982                NaN                NaN   
                  1983                NaN                NaN   

                        casos-faixa_50-59  casos-faixa_60-69  \
REGIONAL DE SAÚDE ANO                                          
alfenas           1979                NaN                NaN   
                  1980                NaN                NaN   
                  1981                NaN                NaN   
                  1982                NaN                NaN   
                  1983                NaN                NaN   

                        casos-faixa_70-ou-mais  casos-faixa_total  \
REGIONAL DE SAÚDE ANO                                               
alfenas           1979                     NaN                NaN   
                  1980                     NaN                NaN   
                  1981                     NaN                NaN   
                  1982                     NaN                NaN   
                  1983                     NaN                NaN   

                        casos-escolaridade_fundamental-completo  ...  \
REGIONAL DE SAÚDE ANO                                            ...   
alfenas           1979                                      NaN  ...   
                  1980                                      NaN  ...   
                  1981                                      NaN  ...   
                  1982                                      NaN  ...   
                  1983                                      NaN  ...   

                        mortalidade_total  mamografia_0-9  mamografia_10-19  \
REGIONAL DE SAÚDE ANO                                                         
alfenas           1979               3.54             NaN               NaN   
                  1980               7.47             NaN               NaN   
                  1981               3.99             NaN               NaN   
                  1982               3.38             NaN               NaN   
                  1983               7.82             NaN               NaN   

                        mamografia_20-29  mamografia_30-39  mamografia_40-49  \
REGIONAL DE SAÚDE ANO                                                          
alfenas           1979               NaN               NaN               NaN   
                  1980               NaN               NaN               NaN   
                  1981               NaN               NaN               NaN   
                  1982               NaN               NaN               NaN   
                  1983               NaN               NaN               NaN   

                        mamografia_50-59  mamografia_60-69  \
REGIONAL DE SAÚDE ANO                                        
alfenas           1979               NaN               NaN   
                  1980               NaN               NaN   
                  1981               NaN               NaN   
                  1982               NaN               NaN   
                  1983               NaN               NaN   

                   

In [3]:
print(
    f"Matriz de entrada: {painel.shape[0]} pares Regional-Ano x {painel.shape[1]} variáveis, "
    f"{painel.index.get_level_values(COLUNA_REGIONAL).nunique()} Regionais de Saúde."
)
pd.set_option("display.max_columns", None)
painel.head(10)

Matriz de entrada: 1316 pares Regional-Ano x 42 variáveis, 28 Regionais de Saúde.


casos-faixa_0-9  casos-faixa_10-19  casos-faixa_20-29  \
REGIONAL DE SAÚDE ANO                                                           
alfenas           1979              NaN                NaN                NaN   
                  1980              NaN                NaN                NaN   
                  1981              NaN                NaN                NaN   
                  1982              NaN                NaN                NaN   
                  1983              NaN                NaN                NaN   
                  1984              NaN                NaN                NaN   
                  1985              NaN                NaN                NaN   
                  1986              NaN                NaN                NaN   
                  1987              NaN                NaN                NaN   
                  1988              NaN                NaN                NaN   

                        casos-faixa_30-39  casos-faixa_40-49  \
REGIONAL DE SAÚDE ANO                                          
alfenas           1979                NaN                NaN   
                  1980                NaN                NaN   
                  1981                NaN                NaN   
                  1982                NaN                NaN   
                  1983                NaN                NaN   
                  1984                NaN                NaN   
                  1985                NaN                NaN   
                  1986                NaN                NaN   
                  1987                NaN                NaN   
                  1988                NaN                NaN   

                        casos-faixa_50-59  casos-faixa_60-69  \
REGIONAL DE SAÚDE ANO                                          
alfenas           1979                NaN                NaN   
                  1980                NaN                NaN   
                  1981                NaN                NaN   
                  1982                NaN                NaN   
                  1983                NaN                NaN   
                  1984                NaN                NaN   
                  1985                NaN                NaN   
                  1986                NaN                NaN   
                  1987                NaN                NaN   
                  1988                NaN                NaN   

                        casos-faixa_70-ou-mais  casos-faixa_total  \
REGIONAL DE SAÚDE ANO                                               
alfenas           1979                     NaN                NaN   
                  1980                     NaN                NaN   
                  1981                     NaN                NaN   
                  1982                     NaN                NaN   
                  1983                     NaN                NaN   
                  1984                     NaN                NaN   
                  1985                     NaN                NaN   
                  1986                     NaN                NaN   
                  1987                     NaN                NaN   
                  1988                     NaN                NaN   

                        casos-escolaridade_fundamental-completo  \
REGIONAL DE SAÚDE ANO                                             
alfenas           1979                                      NaN   
                  1980                                      NaN   
                  1981                                      NaN   
                  1982                                      NaN   
                  1983                                      NaN   
                  1984                                      NaN   
                  1985                                      NaN   
                  1986                                      NaN   
              

In [4]:
matriz_exibicao = descartar_colunas_sem_informacao_e_total(painel).reset_index()
print(
    f"Matriz de entrada (Ano como coluna; sem colunas 'total'/'sem informação'): "
    f"{matriz_exibicao.shape[0]} linhas x {matriz_exibicao.shape[1]} colunas, "
    f"{matriz_exibicao[COLUNA_REGIONAL].nunique()} Regionais de Saúde."
)
matriz_exibicao.head(10)

[LOG] Descartada(s) 7 coluna(s) sem-informação/total: ['casos-faixa_total', 'casos-escolaridade_sem-informacao', 'casos-escolaridade_total', 'casos-raca-cor_sem-informacao', 'casos-raca-cor_total', 'mortalidade_total', 'mamografia_total']
Matriz de entrada (Ano como coluna; sem colunas 'total'/'sem informação'): 1316 linhas x 37 colunas, 28 Regionais de Saúde.


,REGIONAL DE SAÚDE,ANO,casos-faixa_0-9,casos-faixa_10-19,casos-faixa_20-29,casos-faixa_30-39,casos-faixa_40-49,casos-faixa_50-59,casos-faixa_60-69,casos-faixa_70-ou-mais,casos-escolaridade_fundamental-completo,casos-escolaridade_fundamental-incompleto,casos-escolaridade_nenhuma,casos-escolaridade_nivel-medio,casos-escolaridade_nivel-superior-completo,casos-escolaridade_nivel-superior-incompleto,casos-raca-cor_amarela,casos-raca-cor_branca,casos-raca-cor_indigena,casos-raca-cor_parda,casos-raca-cor_preta,mortalidade_0-9,mortalidade_10-19,mortalidade_20-29,mortalidade_30-39,mortalidade_40-49,mortalidade_50-59,mortalidade_60-69,mortalidade_70-ou-mais,mamografia_0-9,mamografia_10-19,mamografia_20-29,mamografia_30-39,mamografia_40-49,mamografia_50-59,mamografia_60-69,mamografia_70-ou-mais
0,alfenas,1979,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.37,28.32,14.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,alfenas,1980,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.96,21.25,55.53,41.03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,alfenas,1981,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.89,6.94,27.26,13.27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,alfenas,1982,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.93,NaN,17.83,NaN,60.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,alfenas,1983,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.82,19.98,8.75,50.06,76.91,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,alfenas,1984,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.05,25.76,36.43,110.58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,alfenas,1985,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.39,42.12,11.78,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,alfenas,1986,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.71,18.78,41.32,11.44,33.85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,alfenas,1987,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.84,6.13,NaN,33.29,64.86,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,alfenas,1988,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.91,43.07,77.70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Janela temporal, descarte de colunas e tratamento de nulos


In [5]:
if USAR_JANELA_COMUM:
    anos = painel.index.get_level_values(COLUNA_ANO)
    matriz = painel[(anos >= ANO_INICIAL_COMUM) & (anos <= ANO_FINAL_COMUM)]
else:
    registrar_log("USAR_JANELA_COMUM=False: usando o período completo de cada bloco, sem restringir ano.")
    matriz = painel

matriz = descartar_colunas_sem_informacao_e_total(matriz)
matriz = descartar_colunas_com_muitos_nulos(matriz, limiar_maximo=LIMIAR_MAXIMO_DE_NULOS_POR_COLUNA)
matriz_antes_do_dropna = matriz
matriz = tratar_nulos_sem_imputar(matriz)

registrar_log(f"Matriz final: {matriz.shape[0]} linha(s) x {matriz.shape[1]} atributo(s).")
matriz.head()

[LOG] Descartada(s) 7 coluna(s) sem-informação/total: ['casos-faixa_total', 'casos-escolaridade_sem-informacao', 'casos-escolaridade_total', 'casos-raca-cor_sem-informacao', 'casos-raca-cor_total', 'mortalidade_total', 'mamografia_total']
[LOG] Descartada(s) 6 coluna(s) com mais de 5% de nulos: ['casos-faixa_0-9', 'casos-faixa_10-19', 'mortalidade_0-9', 'mortalidade_10-19', 'mortalidade_20-29', 'mortalidade_30-39']
[LOG] Descartada(s) 24 de 280 linha(s) (8.6%) por ter ao menos um valor ausente -- nenhum valor foi imputado.
[LOG] Matriz final: 256 linha(s) x 29 atributo(s).


casos-faixa_20-29  casos-faixa_30-39  \
REGIONAL DE SAÚDE ANO                                          
alfenas           2014              11.27              86.41   
                  2015              11.37              76.55   
                  2016              14.35              81.29   
                  2017               5.80              44.51   
                  2018               0.00              49.72   

                        casos-faixa_40-49  casos-faixa_50-59  \
REGIONAL DE SAÚDE ANO                                          
alfenas           2014             116.97             160.58   
                  2015             150.01             204.35   
                  2016             118.78             165.55   
                  2017             109.05             138.38   
                  2018             132.57             118.94   

                        casos-faixa_60-69  casos-faixa_70-ou-mais  \
REGIONAL DE SAÚDE ANO                                               
alfenas           2014             241.86                  261.18   
                  2015             177.21                  251.93   
                  2016             194.96                  283.08   
                  2017             159.53                  214.66   
                  2018             140.00                  147.80   

                        casos-escolaridade_fundamental-completo  \
REGIONAL DE SAÚDE ANO                                             
alfenas           2014                                    20.72   
                  2015                                    23.77   
                  2016                                    19.04   
                  2017                                    26.96   
                  2018                                    34.26   

                        casos-escolaridade_fundamental-incompleto  \
REGIONAL DE SAÚDE ANO                                               
alfenas           2014                                      32.23   
                  2015                                      40.22   
                  2016                                      39.90   
                  2017                                      19.32   
                  2018                                      10.23   

                        casos-escolaridade_nenhuma  \
REGIONAL DE SAÚDE ANO                                
alfenas           2014                        4.14   
                  2015                        4.57   
                  2016                        3.17   
                  2017                        2.70   
                  2018                        1.33   

                        casos-escolaridade_nivel-medio  \
REGIONAL DE SAÚDE ANO                                    
alfenas           2014                           12.43   
                  2015                            9.60   
                  2016                           10.88   
                  2017                           12.58   
                  2018                            9.34   

                        casos-escolaridade_nivel-superior-completo  \
REGIONAL DE SAÚDE ANO                                                
alfenas           2014                                       11.51   
                  2015                                       11.43   
                  2016                                       12.69   
                  2017                                        8.09   
                  2018                                       10.23   

                        casos-escolaridade_nivel-superior-incompleto  \
REGIONAL DE SAÚDE ANO                                                  
alfenas           2014                                          0.46   
                  2015                                          1.37   
                  2016                                          0.45   
                  2017                             

### Diagnóstico: o que foi removido, explicitamente

A etapa acima só loga contagens. Aqui vão as identidades: quais colunas caíram (sem-informação/total + esparsas) e, de cada linha (par Regional-Ano) descartada pelo `dropna`, quais colunas específicas estavam nulas nela -- para conferir que o descarte é coerente com a cobertura real dos CSVs de origem, não um bug de junção.

In [6]:
colunas_descartadas = sorted(set(painel.columns) - set(matriz_antes_do_dropna.columns))
print(f"Colunas descartadas (sem-informação/total + >{LIMIAR_MAXIMO_DE_NULOS_POR_COLUNA:.0%} de nulos): {len(colunas_descartadas)} de {painel.shape[1]}")
display(pd.DataFrame({"COLUNA_DESCARTADA": colunas_descartadas}))

linhas_descartadas_idx = matriz_antes_do_dropna.index.difference(matriz.index)
print(f"\nLinhas (pares Regional-Ano) descartadas pelo dropna: {len(linhas_descartadas_idx)} de {len(matriz_antes_do_dropna)}")

mascara_nulos = matriz_antes_do_dropna.loc[linhas_descartadas_idx].isna()
colunas_nulas_por_linha = mascara_nulos.apply(lambda linha: ", ".join(linha.index[linha]), axis=1)
tabela_linhas_descartadas = colunas_nulas_por_linha.reset_index()
tabela_linhas_descartadas.columns = [COLUNA_REGIONAL, COLUNA_ANO, "COLUNAS_COM_NULO"]
tabela_linhas_descartadas

Colunas descartadas (sem-informação/total + >5% de nulos): 13 de 42


,COLUNA_DESCARTADA
0,casos-escolaridade_sem-informacao
1,casos-escolaridade_total
2,casos-faixa_0-9
3,casos-faixa_10-19
4,casos-faixa_total
5,casos-raca-cor_sem-informacao
6,casos-raca-cor_total
7,mamografia_total
8,mortalidade_0-9
9,mortalidade_10-19



Linhas (pares Regional-Ano) descartadas pelo dropna: 24 de 280


,REGIONAL DE SAÚDE,ANO,COLUNAS_COM_NULO
0,diamantina,2017,mortalidade_40-49
1,diamantina,2020,mortalidade_50-59
2,diamantina,2022,mortalidade_40-49
3,ituiutaba,2017,mortalidade_40-49
4,ituiutaba,2018,mortalidade_50-59
5,ituiutaba,2022,mortalidade_40-49
6,ituiutaba,2023,mortalidade_40-49
7,pedra azul,2014,mortalidade_70-ou-mais
8,pedra azul,2015,mortalidade_40-49
9,pedra azul,2018,mortalidade_50-59


### Diagnóstico: proporção de nulos por coluna e cobertura da mamografia em 2013

Proporção de nulos de cada coluna na janela comum, **antes** do descarte por limiar (já sem as colunas de total e sem informação), e a cobertura da base de mamografia em 2013 -- que justifica iniciar a janela em 2014.

In [7]:
anos_painel = painel.index.get_level_values(COLUNA_ANO)
matriz_janela = descartar_colunas_sem_informacao_e_total(
    painel[(anos_painel >= ANO_INICIAL_COMUM) & (anos_painel <= ANO_FINAL_COMUM)]
)
proporcao_de_nulos = matriz_janela.isna().mean().sort_values(ascending=False)
print(f"Proporção de nulos por coluna em {ANO_INICIAL_COMUM}-{ANO_FINAL_COMUM}, antes do descarte (só colunas com algum nulo):")
display(proporcao_de_nulos[proporcao_de_nulos > 0].rename("PROPORCAO_DE_NULOS").to_frame())

colunas_mamografia = []
for coluna in painel.columns:
    if coluna.startswith("mamografia_") and not coluna.endswith("_total"):
        colunas_mamografia.append(coluna)
mamografia_2013 = painel[anos_painel == 2013][colunas_mamografia]
print(
    f"Mamografia em 2013: {mamografia_2013.notna().any(axis=1).sum()} de {len(mamografia_2013)} "
    "Regionais com algum registro. Proporção de nulos por faixa:"
)
mamografia_2013.isna().mean().rename("PROPORCAO_DE_NULOS_2013").to_frame()

[LOG] Descartada(s) 7 coluna(s) sem-informação/total: ['casos-faixa_total', 'casos-escolaridade_sem-informacao', 'casos-escolaridade_total', 'casos-raca-cor_sem-informacao', 'casos-raca-cor_total', 'mortalidade_total', 'mamografia_total']
Proporção de nulos por coluna em 2014-2023, antes do descarte (só colunas com algum nulo):


,PROPORCAO_DE_NULOS
mortalidade_0-9,1.000000
mortalidade_10-19,0.992857
mortalidade_20-29,0.767857
casos-faixa_0-9,0.400000
mortalidade_30-39,0.132143
casos-faixa_10-19,0.100000
mortalidade_40-49,0.042857
mortalidade_50-59,0.021429
mortalidade_70-ou-mais,0.010714
mortalidade_60-69,0.007143


Mamografia em 2013: 3 de 28 Regionais com algum registro. Proporção de nulos por faixa:


,PROPORCAO_DE_NULOS_2013
mamografia_0-9,1.000000
mamografia_10-19,1.000000
mamografia_20-29,1.000000
mamografia_30-39,0.892857
mamografia_40-49,0.892857
mamografia_50-59,0.892857
mamografia_60-69,0.892857
mamografia_70-ou-mais,0.892857


## 3. Padronização (RobustScaler)

Cada variável é centrada na mediana e dividida pela amplitude interquartil (IQR). O KMeans roda diretamente sobre as 29 variáveis padronizadas.

In [8]:
matriz_padronizada, matriz_para_kmeans, pca, variancia_acumulada = padronizar_e_reduzir(
    matriz,
    razao_maxima_atributos_por_linha=RAZAO_MAXIMA_ATRIBUTOS_POR_LINHA,
    variancia_alvo=VARIANCIA_EXPLICADA_ALVO,
    numero_maximo_de_componentes=NUMERO_MAXIMO_DE_COMPONENTES,
    semente=SEMENTE_ALEATORIA,
    usar_pca=USAR_PCA,
)

[LOG] PCA não aplicado: USAR_PCA=False (forçado; razão atributos/linhas = 0.113).


## 4. Escolha de k (cotovelo da curva de inércia)

In [9]:
tabela_k = calcular_inercia_e_silhueta_por_k(matriz_para_kmeans, k_minimo=K_MINIMO, k_maximo=K_MAXIMO, semente=SEMENTE_ALEATORIA)
k_escolhido, indice_do_cotovelo = escolher_k_pelo_cotovelo(tabela_k)

figura_cotovelo = grafico_cotovelo_inercia(tabela_k, indice_do_cotovelo, k_escolhido)
figura_cotovelo.show()
tabela_k.drop(columns="silhueta")

[LOG] Inércia calculada(s) para k = 2..12 (11 valor(es) de k).
[LOG] Cotovelo da curva de inércia na posição 3 de 11 -- k = 5


,k,inercia,n_iter_
0,2,5151.241069,8
1,3,4492.131216,11
2,4,4060.881057,14
3,5,3743.330815,12
4,6,3560.265650,7
5,7,3417.892652,13
6,8,3266.809198,19
7,9,3140.871574,15
8,10,3051.904629,13
9,11,2946.795863,21


### Busca em grade

In [10]:
tabela_da_busca = busca_em_grade_k(matriz_para_kmeans, k_minimo=K_MINIMO, k_maximo=K_MAXIMO, sementes=SEMENTES_DA_BUSCA)
print(f"Resultados da busca em grade: k = {K_MINIMO}..{min(K_MAXIMO, len(matriz_para_kmeans) - 1)}, sementes = {SEMENTES_DA_BUSCA} ({len(tabela_da_busca)} combinações).")

resumo_por_k = tabela_da_busca.groupby("k").agg(
    inercia_media=("inercia", "mean"), inercia_desvio=("inercia", "std"),
).reset_index()
print("\nEstabilidade por k através das sementes testadas:")
resumo_por_k

Resultados da busca em grade: k = 2..12, sementes = [0, 10, 30, 42, 66] (55 combinações).

Estabilidade por k através das sementes testadas:


,k,inercia_media,inercia_desvio
0,2,5151.251911,0.024243
1,3,4494.026089,2.580797
2,4,4063.223968,2.715858
3,5,3740.155592,2.631758
4,6,3561.905198,7.684126
5,7,3408.567818,9.962383
6,8,3250.383336,9.830354
7,9,3133.897235,16.623207
8,10,3038.308178,23.750363
9,11,2945.086049,12.976049


In [11]:
print(f"Variação entre sementes para o k escolhido (k = {k_escolhido}):")
tabela_da_busca[tabela_da_busca["k"] == k_escolhido][["inercia"]].agg(["min", "max"])

Variação entre sementes para o k escolhido (k = 5):


,inercia
min,3738.088537
max,3743.330815


In [12]:
inercia_por_semente = tabela_da_busca.loc[tabela_da_busca["k"] == k_escolhido, "inercia"]
desvio_inercia = inercia_por_semente.std()
media_inercia = inercia_por_semente.mean()
cv_inercia = desvio_inercia / media_inercia * 100

print(f"Coeficiente de variação da inércia entre as {len(inercia_por_semente)} sementes (k = {k_escolhido}):")
print(f"  S  = {desvio_inercia:.4f}")
print(f"  x̄  = {media_inercia:.4f}")
print(f"  CV = {cv_inercia:.4f}%")

Coeficiente de variação da inércia entre as 5 sementes (k = 5):
  S  = 2.6318
  x̄  = 3740.1556
  CV = 0.0704%


In [13]:
maximo_iteracoes = maximo_de_iteracoes_usadas(tabela_k, tabela_da_busca)
max_iter_configurado = MAXIMO_DE_ITERACOES
print(
    f"Máximo de iterações até convergência entre todas as execuções de KMeans "
    f"(cotovelo + busca em grade): {maximo_iteracoes} (max_iter configurado: {max_iter_configurado})."
)


Máximo de iterações até convergência entre todas as execuções de KMeans (cotovelo + busca em grade): 29 (max_iter configurado: 30).


## 5. KMeans

In [14]:
rotulos, modelo_kmeans = executar_kmeans(matriz_para_kmeans, k_escolhido, semente=SEMENTE_ALEATORIA)
n_clusters, inercia_final, silhueta_final = resumir_agrupamento_kmeans(matriz_para_kmeans, rotulos, modelo_kmeans)

registrar_log(f"KMeans(k={k_escolhido}, random_state={SEMENTE_ALEATORIA})")
registrar_log(f"  Agrupamentos: {n_clusters}")
registrar_log(f"  Inércia (WCSS): {inercia_final:.4f}")

_ = diagnostico_tamanho_dos_clusters(rotulos, minimo=MINIMO_DE_ELEMENTOS_POR_CLUSTER)

[LOG] KMeans(k=5, random_state=42)
[LOG]   Agrupamentos: 5
[LOG]   Inércia (WCSS): 3743.3308


## 6. Caracterização dos clusters — como o cálculo é feito

A matriz usada pelo KMeans (`matriz_para_kmeans`) foi padronizada pelo `RobustScaler` -- seus valores não são diretamente interpretáveis (não estão na unidade original de taxa por 100 mil habitantes). Por isso "o que caracteriza cada cluster" é calculado à parte, sobre `matriz` (as variáveis originais, antes de qualquer padronização), cruzada com os rótulos (`rotulos`) que o KMeans devolveu para cada linha. O cálculo, passo a passo (`calcular_caracterizacao_por_cluster`, em `utilitarios_agrupamento_kmeans.py`):

1. **Média geral de cada variável.** Para cada coluna de `matriz` (cada taxa original -- ex. `mortalidade_60-69`), calcula-se a média sobre TODAS as linhas da base, sem agrupar por cluster. Essa é a referência ("perfil médio") contra a qual cada cluster é comparado.
2. **Média da mesma variável, dentro de cada cluster.** As linhas de `matriz` são agrupadas pelo `CLUSTER` de cada par Regional-Ano (`montar_tabela_pertencimento` faz essa junção) e, dentro de cada grupo, calcula-se a média de cada variável -- ou seja, o valor médio daquela taxa só entre os pares que caíram naquele cluster.
3. **Diferença padronizada pelo IQR geral** (Everitt et al., 2011, Seção 5.6.2). Para cada par (cluster, variável):
   ```
   DIFERENCA_PADRONIZADA = (MEDIA_CLUSTER - MEDIA_GERAL) / IQR_GERAL
   ```
   `IQR_GERAL` usa o mesmo Q1/Q3 do `RobustScaler`, então a diferença fica na mesma escala em que o KMeans calculou as distâncias. Positivo = cluster ACIMA da média geral; negativo = ABAIXO. Variáveis com IQR geral 0 ficam `NaN` e vão para o fim da ordenação. A `DIFERENCA_PERCENTUAL` = (MEDIA_CLUSTER - MEDIA_GERAL) / |MEDIA_GERAL| × 100 continua na tabela **só como informativa** -- ela explode quando a média geral é próxima de 0 (categorias raras), por isso não é usada como critério.
4. **Ordenação por relevância.** Dentro de cada cluster, as variáveis são ordenadas pelo valor ABSOLUTO da diferença padronizada, do maior para o menor -- as primeiras linhas são as que mais afastam aquele cluster do perfil médio geral. O sinal (acima/abaixo) não entra no critério de ordenação, só no resultado. Não há teste de significância (ANOVA/Tukey): seria circular, pois o KMeans maximiza a diferença entre grupos nessas mesmas variáveis (Everitt et al., 2011, Seção 9.2, item ix).
5. **Seleção das top-N.** `selecionar_variaveis_mais_discriminantes` pega, de cada cluster já ordenado, só as `NUMERO_DE_VARIAVEIS_DISCRIMINANTES` primeiras linhas -- as variáveis mais discriminantes daquele cluster especificamente (não da base como um todo).
6. **Texto por cluster.** `descrever_caracterizacao_por_cluster` converte essas top-N linhas em uma frase por cluster, no formato "**VARIÁVEL** acima/abaixo da média geral em X IQR (±Y%)", na mesma ordem de relevância -- é a versão em texto corrido da tabela, para leitura direta sem precisar cruzar números manualmente.

Nenhuma etapa interpreta clinicamente a variável (não decide se "mortalidade alta" é bom ou ruim, nem nomeia um "perfil" para o cluster) -- isso é leitura humana, feita na Parte 2 a partir do resultado numérico.

# Parte 2 — Avaliação e Apresentação (Resultados)


## Parâmetros finais e atributos usados

In [15]:
resumo_parametros = montar_resumo_parametros_kmeans(
    k_escolhido, inercia_final, silhueta_final,
    usar_robust_scaler=True, pca=pca, variancia_acumulada=variancia_acumulada,
    n_linhas=matriz.shape[0], n_atributos=matriz.shape[1], n_clusters=n_clusters,
)
resumo_parametros.drop(columns=["PCA_APLICADO", "PCA_N_COMPONENTES", "PCA_VARIANCIA_EXPLICADA", "SILHUETA"])

,K,ESCALONADOR,N_LINHAS,N_ATRIBUTOS,N_CLUSTERS,INERCIA
0,5,RobustScaler,256,29,5,3743.330815


In [16]:
atributos_usados = pd.DataFrame({"ATRIBUTO": list(matriz.columns)})
atributos_usados

,ATRIBUTO
0,casos-faixa_20-29
1,casos-faixa_30-39
2,casos-faixa_40-49
3,casos-faixa_50-59
4,casos-faixa_60-69
5,casos-faixa_70-ou-mais
6,casos-escolaridade_fundamental-completo
7,casos-escolaridade_fundamental-incompleto
8,casos-escolaridade_nenhuma
9,casos-escolaridade_nivel-medio


## Visualizações

In [17]:
resultado = montar_tabela_pertencimento(matriz.index, rotulos)

figura_regional = grafico_distribuicao_por_regional(resultado)
figura_regional.show()

figura_ano = grafico_distribuicao_por_ano(resultado)
figura_ano.show()

### Composição de cada cluster (tabela, não só o gráfico)


In [18]:
resumo_por_cluster = montar_resumo_por_cluster(resultado)
resumo_por_cluster

,CLUSTER,N_PARES,N_REGIONAIS_DISTINTAS,MEMBROS
0,1,45,16,"alfenas (2017), alfenas (2019), barbacena (201..."
1,2,61,13,"diamantina (2014), diamantina (2015), diamanti..."
2,3,97,19,"barbacena (2014), barbacena (2015), barbacena ..."
3,4,49,12,"alfenas (2014), alfenas (2015), alfenas (2016)..."
4,5,4,1,"passos (2019), passos (2021), passos (2022), p..."


## Mapa: cluster por Regional e Ano

In [19]:
resultado

,REGIONAL DE SAÚDE,ANO,CLUSTER
0,alfenas,2014,4
1,alfenas,2015,4
2,alfenas,2016,4
3,alfenas,2017,1
4,alfenas,2018,4
...,...,...,...
251,varginha,2019,3
252,varginha,2020,3
253,varginha,2021,3
254,varginha,2022,3


In [20]:
visao_agregada_regional_ano = montar_visao_agregada_regional_ano(resultado)
visao_agregada_regional_ano

ANO,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
REGIONAL DE SAÚDE,,,,,,,,,,
alfenas,4.0,4.0,4.0,1.0,4.0,1.0,4.0,4.0,4.0,4.0
barbacena,3.0,3.0,1.0,4.0,1.0,1.0,3.0,3.0,1.0,4.0
belo horizonte,3.0,3.0,3.0,1.0,1.0,1.0,3.0,3.0,3.0,3.0
coronel fabriciano,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
diamantina,2.0,2.0,2.0,NaN,2.0,2.0,NaN,2.0,NaN,2.0
divinopolis,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,4.0,3.0
governador valadares,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
itabira,3.0,2.0,1.0,1.0,2.0,2.0,3.0,3.0,4.0,4.0
ituiutaba,2.0,3.0,3.0,NaN,NaN,3.0,2.0,2.0,NaN,NaN


### Heatmap: cluster por Regional e Ano

In [21]:
figura_heatmap = grafico_heatmap_regional_ano(visao_agregada_regional_ano)
figura_heatmap.show()

## Caracterização dos clusters — resultado

O método deste cálculo está na Parte 1, Seção 6. Aqui só o resultado: a tabela com a média de cada variável por cluster (comparada à média geral), o gráfico das variáveis mais discriminantes e, por fim, um resumo em texto corrido do que caracteriza cada cluster.

In [22]:
caracterizacao_por_cluster = calcular_caracterizacao_por_cluster(matriz, rotulos)
variaveis_discriminantes = selecionar_variaveis_mais_discriminantes(
    caracterizacao_por_cluster, top_n=NUMERO_DE_VARIAVEIS_DISCRIMINANTES
)

for cluster in sorted(variaveis_discriminantes["CLUSTER"].unique()):
    print(f"\n=== Cluster {cluster} — variáveis mais discriminantes ===")
    display(
        variaveis_discriminantes[variaveis_discriminantes["CLUSTER"] == cluster]
        .drop(columns="CLUSTER")
        .reset_index(drop=True)
    )

[LOG] ATENÇÃO: 1 variável(is) com IQR geral = 0 (sem diferença padronizada): ['casos-raca-cor_indigena']
[LOG] Caracterização por cluster: 5 cluster(s) x 29 variável(is).

=== Cluster 1 — variáveis mais discriminantes ===


,VARIAVEL,MEDIA_CLUSTER,MEDIA_GERAL,DIFERENCA_PADRONIZADA,DIFERENCA_PERCENTUAL
0,mamografia_30-39,1161.938667,567.442656,1.439405,104.767593
1,mamografia_70-ou-mais,4063.642889,1961.735898,1.318291,107.145258
2,mamografia_40-49,9015.010444,4729.165039,1.262474,90.625837
3,mamografia_60-69,16334.602444,10577.956289,0.832235,54.421157
4,mamografia_50-59,19250.696000,12687.698516,0.782311,51.727250
5,mamografia_10-19,12.492667,6.289961,0.780951,98.612786
6,mamografia_20-29,89.638222,52.529297,0.723971,70.644245
7,mamografia_0-9,9.049111,5.404258,0.474281,67.444105
8,casos-faixa_20-29,3.334889,5.372773,-0.378437,-37.929843
9,casos-escolaridade_fundamental-completo,8.008000,10.104492,-0.299392,-20.748120



=== Cluster 2 — variáveis mais discriminantes ===


,VARIAVEL,MEDIA_CLUSTER,MEDIA_GERAL,DIFERENCA_PADRONIZADA,DIFERENCA_PERCENTUAL
0,casos-faixa_60-69,90.563279,162.991250,-1.046193,-44.436724
1,casos-faixa_70-ou-mais,81.316066,163.081602,-0.869130,-50.137805
2,casos-faixa_50-59,88.730328,138.746680,-0.862648,-36.048684
3,casos-escolaridade_fundamental-completo,4.839016,10.104492,-0.751942,-52.110247
4,casos-faixa_40-49,63.793279,95.669062,-0.737652,-33.318800
5,casos-escolaridade_nivel-medio,6.782295,12.836328,-0.721792,-47.163277
6,casos-escolaridade_fundamental-incompleto,11.840000,21.388008,-0.660876,-44.641875
7,mortalidade_70-ou-mais,39.803770,62.208164,-0.654860,-36.015198
8,casos-escolaridade_nivel-superior-incompleto,0.339672,1.064453,-0.651489,-68.089517
9,casos-raca-cor_branca,9.640656,29.861523,-0.649041,-67.715459



=== Cluster 3 — variáveis mais discriminantes ===


,VARIAVEL,MEDIA_CLUSTER,MEDIA_GERAL,DIFERENCA_PADRONIZADA,DIFERENCA_PERCENTUAL
0,mamografia_40-49,3066.957113,4729.165039,-0.489634,-35.148021
1,mamografia_30-39,396.449794,567.442656,-0.414011,-30.133946
2,mamografia_50-59,9528.978454,12687.698516,-0.376520,-24.895926
3,mamografia_60-69,8035.951649,10577.956289,-0.367496,-24.031151
4,mamografia_20-29,34.545155,52.529297,-0.350859,-34.236404
5,mamografia_70-ou-mais,1417.431753,1961.735898,-0.341381,-27.746046
6,casos-escolaridade_nivel-superior-incompleto,0.776495,1.064453,-0.258839,-27.052227
7,mamografia_10-19,4.404021,6.289961,-0.237449,-29.983339
8,casos-escolaridade_nenhuma,3.586907,3.012734,0.224945,19.058197
9,mortalidade_70-ou-mais,69.596289,62.208164,0.215948,11.876455



=== Cluster 4 — variáveis mais discriminantes ===


,VARIAVEL,MEDIA_CLUSTER,MEDIA_GERAL,DIFERENCA_PADRONIZADA,DIFERENCA_PERCENTUAL
0,casos-escolaridade_fundamental-completo,17.889388,10.104492,1.111731,77.043907
1,casos-escolaridade_nivel-medio,21.945510,12.836328,1.086043,70.964079
2,casos-raca-cor_preta,9.399184,4.982734,1.017029,88.635054
3,casos-faixa_40-49,137.156122,95.669062,0.960071,43.365179
4,casos-escolaridade_nivel-superior-completo,14.100816,8.083867,0.950920,74.431568
5,casos-raca-cor_branca,58.066531,29.861523,0.905312,94.452673
6,casos-faixa_50-59,190.947347,138.746680,0.900322,37.623003
7,casos-faixa_60-69,224.016531,162.991250,0.881486,37.440832
8,casos-faixa_70-ou-mais,235.916122,163.081602,0.774197,44.661397
9,casos-raca-cor_amarela,0.753673,0.368437,0.770472,104.559381



=== Cluster 5 — variáveis mais discriminantes ===


,VARIAVEL,MEDIA_CLUSTER,MEDIA_GERAL,DIFERENCA_PADRONIZADA,DIFERENCA_PERCENTUAL
0,casos-escolaridade_nivel-superior-incompleto,14.0825,1.064453,11.701615,1222.979817
1,mamografia_40-49,21030.8825,4729.165039,4.801967,344.706039
2,mamografia_20-29,218.6475,52.529297,3.240857,316.239152
3,mamografia_10-19,23.8750,6.289961,2.214043,279.573104
4,mamografia_30-39,1410.3775,567.442656,2.040930,148.549785
5,casos-escolaridade_nivel-medio,28.3500,12.836328,1.849618,120.857552
6,casos-escolaridade_fundamental-completo,0.1200,10.104492,-1.425847,-98.812409
7,mamografia_70-ou-mais,3846.2250,1961.735898,1.181930,96.062324
8,mamografia_50-59,21814.2950,12687.698516,1.087892,71.932640
9,mamografia_60-69,17805.2200,10577.956289,1.044841,68.323819


In [23]:
figura_discriminantes = grafico_variaveis_discriminantes_por_cluster(variaveis_discriminantes)
figura_discriminantes.show()

### Resumo em texto: o que caracteriza cada cluster

Mesma informação da tabela e do gráfico acima, em texto corrido -- as variáveis mais discriminantes de cada cluster, na ordem de relevância, com o sinal, a diferença em IQR e o percentual (informativo) em relação à média geral da base.

In [24]:
from IPython.display import Markdown

display(Markdown(descrever_caracterizacao_por_cluster(caracterizacao_por_cluster, resumo_por_cluster)))

- **Cluster 1** (45 pares Regional-Ano): **mamografia_30-39** acima da média geral em 1.44 IQR (+104.8%); **mamografia_70-ou-mais** acima da média geral em 1.32 IQR (+107.1%); **mamografia_40-49** acima da média geral em 1.26 IQR (+90.6%); **mamografia_60-69** acima da média geral em 0.83 IQR (+54.4%); **mamografia_50-59** acima da média geral em 0.78 IQR (+51.7%); **mamografia_10-19** acima da média geral em 0.78 IQR (+98.6%); **mamografia_20-29** acima da média geral em 0.72 IQR (+70.6%); **mamografia_0-9** acima da média geral em 0.47 IQR (+67.4%); **casos-faixa_20-29** abaixo da média geral em 0.38 IQR (-37.9%); **casos-escolaridade_fundamental-completo** abaixo da média geral em 0.30 IQR (-20.7%); **mortalidade_70-ou-mais** acima da média geral em 0.30 IQR (+16.2%); **casos-raca-cor_amarela** abaixo da média geral em 0.25 IQR (-33.5%); **casos-faixa_40-49** abaixo da média geral em 0.19 IQR (-8.5%); **mortalidade_60-69** acima da média geral em 0.18 IQR (+10.3%); **casos-raca-cor_parda** abaixo da média geral em 0.15 IQR (-12.3%).
- **Cluster 2** (61 pares Regional-Ano): **casos-faixa_60-69** abaixo da média geral em 1.05 IQR (-44.4%); **casos-faixa_70-ou-mais** abaixo da média geral em 0.87 IQR (-50.1%); **casos-faixa_50-59** abaixo da média geral em 0.86 IQR (-36.0%); **casos-escolaridade_fundamental-completo** abaixo da média geral em 0.75 IQR (-52.1%); **casos-faixa_40-49** abaixo da média geral em 0.74 IQR (-33.3%); **casos-escolaridade_nivel-medio** abaixo da média geral em 0.72 IQR (-47.2%); **casos-escolaridade_fundamental-incompleto** abaixo da média geral em 0.66 IQR (-44.6%); **mortalidade_70-ou-mais** abaixo da média geral em 0.65 IQR (-36.0%); **casos-escolaridade_nivel-superior-incompleto** abaixo da média geral em 0.65 IQR (-68.1%); **casos-raca-cor_branca** abaixo da média geral em 0.65 IQR (-67.7%); **casos-escolaridade_nivel-superior-completo** abaixo da média geral em 0.65 IQR (-50.7%); **casos-raca-cor_preta** abaixo da média geral em 0.63 IQR (-55.0%); **casos-faixa_30-39** abaixo da média geral em 0.62 IQR (-38.7%); **mamografia_70-ou-mais** abaixo da média geral em 0.56 IQR (-45.6%); **mamografia_30-39** abaixo da média geral em 0.53 IQR (-38.9%).
- **Cluster 3** (97 pares Regional-Ano): **mamografia_40-49** abaixo da média geral em 0.49 IQR (-35.1%); **mamografia_30-39** abaixo da média geral em 0.41 IQR (-30.1%); **mamografia_50-59** abaixo da média geral em 0.38 IQR (-24.9%); **mamografia_60-69** abaixo da média geral em 0.37 IQR (-24.0%); **mamografia_20-29** abaixo da média geral em 0.35 IQR (-34.2%); **mamografia_70-ou-mais** abaixo da média geral em 0.34 IQR (-27.7%); **casos-escolaridade_nivel-superior-incompleto** abaixo da média geral em 0.26 IQR (-27.1%); **mamografia_10-19** abaixo da média geral em 0.24 IQR (-30.0%); **casos-escolaridade_nenhuma** acima da média geral em 0.22 IQR (+19.1%); **mortalidade_70-ou-mais** acima da média geral em 0.22 IQR (+11.9%); **casos-faixa_60-69** acima da média geral em 0.21 IQR (+8.9%); **mamografia_0-9** abaixo da média geral em 0.19 IQR (-27.7%); **casos-raca-cor_parda** acima da média geral em 0.19 IQR (+15.8%); **mortalidade_60-69** acima da média geral em 0.18 IQR (+10.2%); **casos-escolaridade_nivel-medio** abaixo da média geral em 0.17 IQR (-11.4%).
- **Cluster 4** (49 pares Regional-Ano): **casos-escolaridade_fundamental-completo** acima da média geral em 1.11 IQR (+77.0%); **casos-escolaridade_nivel-medio** acima da média geral em 1.09 IQR (+71.0%); **casos-raca-cor_preta** acima da média geral em 1.02 IQR (+88.6%); **casos-faixa_40-49** acima da média geral em 0.96 IQR (+43.4%); **casos-escolaridade_nivel-superior-completo** acima da média geral em 0.95 IQR (+74.4%); **casos-raca-cor_branca** acima da média geral em 0.91 IQR (+94.5%); **casos-faixa_50-59** acima da média geral em 0.90 IQR (+37.6%); **casos-faixa_60-69** acima da média geral em 0.88 IQR (+37.4%); **casos-faixa_70-ou-mais** acima da média geral em 0.77 IQR (+44.7%); **casos-raca-cor_amarela** acima da média geral em 0.77 IQR (+104.6%); **casos-escolaridade_fundamental-incompleto** acima da média geral em 0.74 IQR (+50.3%); **casos-faixa_30-39** acima da média geral em 0.68 IQR (+42.1%); **casos-faixa_20-29** acima da média geral em 0.50 IQR (+50.5%); **casos-escolaridade_nivel-superior-incompleto** acima da média geral em 0.46 IQR (+47.7%); **mamografia_0-9** acima da média geral em 0.33 IQR (+46.8%).
- **Cluster 5** (4 pares Regional-Ano): **casos-escolaridade_nivel-superior-incompleto** acima da média geral em 11.70 IQR (+1223.0%); **mamografia_40-49** acima da média geral em 4.80 IQR (+344.7%); **mamografia_20-29** acima da média geral em 3.24 IQR (+316.2%); **mamografia_10-19** acima da média geral em 2.21 IQR (+279.6%); **mamografia_30-39** acima da média geral em 2.04 IQR (+148.5%); **casos-escolaridade_nivel-medio** acima da média geral em 1.85 IQR (+120.9%); **casos-escolaridade_fundamental-completo** abaixo da média geral em 1.43 IQR (-98.8%); **mamografia_70-ou-mais** acima da média geral em 1.18 IQR (+96.1%); **mamografia_50-59** acima da média geral em 1.09 IQR (+71.9%); **mamografia_60-69** acima da média geral em 1.04 IQR (+68.3%); **mamografia_0-9** acima da média geral em 0.92 IQR (+130.7%); **casos-faixa_20-29** acima da média geral em 0.71 IQR (+71.3%); **casos-faixa_40-49** acima da média geral em 0.62 IQR (+27.8%); **casos-escolaridade_fundamental-incompleto** acima da média geral em 0.59 IQR (+39.7%); **casos-faixa_50-59** acima da média geral em 0.58 IQR (+24.3%).

## Caracterização individual — decomposição da distância ao centroide

A caracterização por médias (acima) descreve cada cluster como um todo. Aqui, a distância² de cada par Regional-Ano até o centroide do seu cluster -- a mesma que o KMeans usa -- é decomposta em uma parcela por variável, `(valor padronizado − centroide)²` (`calcular_parcelas_distancia_ao_centroide`). A soma das parcelas de um par é a distância² dele; a soma de todas é a inércia final.

In [25]:
parcelas_distancia = calcular_parcelas_distancia_ao_centroide(matriz_para_kmeans, modelo_kmeans, matriz.index, matriz.columns)
assert np.isclose(parcelas_distancia["PARCELA_DISTANCIA"].sum(), inercia_final), "soma das parcelas != inércia final"
print(f"Soma de todas as parcelas = {parcelas_distancia['PARCELA_DISTANCIA'].sum():.4f} | inércia final = {inercia_final:.4f}")

distancia_por_par = (
    parcelas_distancia.groupby([COLUNA_REGIONAL, COLUNA_ANO, "CLUSTER"])["PARCELA_DISTANCIA"].sum()
    .rename("DISTANCIA2").sort_values(ascending=False).reset_index()
)
print("\nPares mais distantes do centroide do seu cluster:")
distancia_por_par.head(10)

Soma de todas as parcelas = 3743.3308 | inércia final = 3743.3308

Pares mais distantes do centroide do seu cluster:


,REGIONAL DE SAÚDE,ANO,CLUSTER,DISTANCIA2
0,passos,2017,1,97.855951
1,leopoldina,2019,4,71.607436
2,patos de minas,2023,4,66.738562
3,alfenas,2018,4,62.068865
4,sete lagoas,2022,4,61.046788
5,coronel fabriciano,2018,3,60.621570
6,passos,2018,1,58.376963
7,alfenas,2023,4,42.208420
8,alfenas,2020,4,38.007866
9,passos,2019,5,37.978906


### Variáveis que mais pesam na distância dos pares mais distantes

Para os quatro pares mais distantes: as três maiores parcelas, a participação de cada uma na distância² do par e o valor original da taxa, comparado à média e à mediana gerais.

In [26]:
for _, par in distancia_por_par.head(4).iterrows():
    regional, ano = par[COLUNA_REGIONAL], par[COLUNA_ANO]
    maiores = parcelas_distancia[
        (parcelas_distancia[COLUNA_REGIONAL] == regional) & (parcelas_distancia[COLUNA_ANO] == ano)
    ].head(3).copy()
    maiores["PARTICIPACAO_NO_PAR"] = maiores["PARCELA_DISTANCIA"] / par["DISTANCIA2"]
    maiores["VALOR_ORIGINAL"] = maiores["VARIAVEL"].map(matriz.loc[(regional, ano)])
    maiores["MEDIA_GERAL"] = maiores["VARIAVEL"].map(matriz.mean())
    maiores["MEDIANA_GERAL"] = maiores["VARIAVEL"].map(matriz.median())
    print(f"\n{regional} {ano} -- Cluster {par['CLUSTER']}, distância² = {par['DISTANCIA2']:.2f}")
    display(maiores.drop(columns=[COLUNA_REGIONAL, COLUNA_ANO, "CLUSTER"]).reset_index(drop=True))


passos 2017 -- Cluster 1, distância² = 97.86


,VARIAVEL,PARCELA_DISTANCIA,PARTICIPACAO_NO_PAR,VALOR_ORIGINAL,MEDIA_GERAL,MEDIANA_GERAL
0,mamografia_40-49,17.963633,0.183572,23403.37,4729.165039,3865.745
1,mamografia_10-19,14.996258,0.153248,43.25,6.289961,4.455
2,mamografia_70-ou-mais,14.167458,0.144779,10064.98,1961.735898,1620.410



leopoldina 2019 -- Cluster 4, distância² = 71.61


,VARIAVEL,PARCELA_DISTANCIA,PARTICIPACAO_NO_PAR,VALOR_ORIGINAL,MEDIA_GERAL,MEDIANA_GERAL
0,mamografia_0-9,17.767983,0.248130,40.33,5.404258,3.050
1,casos-raca-cor_preta,12.447558,0.173831,24.72,4.982734,3.990
2,casos-escolaridade_nenhuma,12.286756,0.171585,12.36,3.012734,2.405



patos de minas 2023 -- Cluster 4, distância² = 66.74


,VARIAVEL,PARCELA_DISTANCIA,PARTICIPACAO_NO_PAR,VALOR_ORIGINAL,MEDIA_GERAL,MEDIANA_GERAL
0,casos-faixa_60-69,21.537143,0.322709,545.30,162.991250,161.775
1,casos-faixa_50-59,13.958904,0.209158,407.57,138.746680,133.600
2,casos-raca-cor_parda,6.560149,0.098296,80.93,25.235977,21.895



alfenas 2018 -- Cluster 4, distância² = 62.07


,VARIAVEL,PARCELA_DISTANCIA,PARTICIPACAO_NO_PAR,VALOR_ORIGINAL,MEDIA_GERAL,MEDIANA_GERAL
0,casos-raca-cor_amarela,31.501874,0.507531,3.56,0.368437,0.00
1,casos-escolaridade_fundamental-completo,5.465421,0.088054,34.26,10.104492,7.99
2,casos-raca-cor_preta,3.078486,0.049598,1.78,4.982734,3.99


### Participação de cada variável na inércia total e homogeneidade dos clusters

In [27]:
participacao_na_inercia = (
    parcelas_distancia.groupby("VARIAVEL")["PARCELA_DISTANCIA"].sum() / inercia_final
).sort_values(ascending=False).rename("PARTICIPACAO_NA_INERCIA")
print("Variáveis que mais contribuem para a inércia total:")
display(participacao_na_inercia.head(10).to_frame())

print("Distância² ao centroide por cluster (menor média = cluster mais homogêneo):")
distancia_por_par.groupby("CLUSTER")["DISTANCIA2"].agg(["mean", "median", "max"])

Variáveis que mais contribuem para a inércia total:


,PARTICIPACAO_NA_INERCIA
VARIAVEL,
casos-raca-cor_amarela,0.080085
casos-escolaridade_nenhuma,0.063201
casos-escolaridade_fundamental-completo,0.059420
mamografia_0-9,0.052383
casos-escolaridade_nivel-superior-incompleto,0.051220
casos-faixa_20-29,0.045536
casos-raca-cor_parda,0.043318
casos-raca-cor_preta,0.039440
mamografia_10-19,0.038057


Distância² ao centroide por cluster (menor média = cluster mais homogêneo):


,mean,median,max
CLUSTER,,,
1,16.625242,12.707932,97.855951
2,9.151010,8.325732,22.290781
3,12.386635,10.854028,60.621570
4,23.552172,18.741528,71.607436
5,20.355811,15.116497,37.978906


### Outros valores citados no texto da monografia

Categorias de escolaridade do Cluster 1 (todas, não só as do top-10) e a trajetória de Passos nas variáveis que definem o Cluster 5.

In [28]:
escolaridade_cluster_1 = caracterizacao_por_cluster[
    (caracterizacao_por_cluster["CLUSTER"] == 1)
    & caracterizacao_por_cluster["VARIAVEL"].str.startswith("casos-escolaridade")
]
display(escolaridade_cluster_1.reset_index(drop=True))

matriz.xs("passos", level=COLUNA_REGIONAL)[["casos-escolaridade_nivel-superior-incompleto", "mamografia_40-49"]]

,CLUSTER,VARIAVEL,MEDIA_CLUSTER,MEDIA_GERAL,DIFERENCA_PADRONIZADA,DIFERENCA_PERCENTUAL
0,1,casos-escolaridade_fundamental-completo,8.008000,10.104492,-0.299392,-20.748120
1,1,casos-escolaridade_nivel-superior-incompleto,0.957333,1.064453,-0.096287,-10.063364
2,1,casos-escolaridade_nenhuma,2.929333,3.012734,-0.032674,-2.768284
3,1,casos-escolaridade_fundamental-incompleto,21.801333,21.388008,0.028609,1.932511
4,1,casos-escolaridade_nivel-superior-completo,8.007556,8.083867,-0.012060,-0.943999
5,1,casos-escolaridade_nivel-medio,12.886667,12.836328,0.006002,0.392157


,casos-escolaridade_nivel-superior-incompleto,mamografia_40-49
ANO,,
2014,0.00,1228.91
2015,0.00,3735.71
2016,2.08,14825.19
2017,0.00,23403.37
2018,5.10,16816.91
2019,8.56,16786.69
2020,3.48,9399.47
2021,13.75,21074.69
2022,17.31,21956.46
